# COS30049 A2 - Explorer Assistant 3B: QLoRA fine-tune of Qwen2.5-3B-Instruct on Kaggle

Runs `lm.qlora` from the project repository ([Defaul119/COS30049A2](https://github.com/Defaul119/COS30049A2)) at a fixed
commit, with the same config, data and code as the local GPU queue (steps `qwen-bench`, `qwen-train`, `eval-qwen` of
`configs/queue_after_pretraining.json`). Moved here on 24 Oct 2026 so the home GPU can train TxGPT and the 152M
stages at the same time.

**Settings (right-hand panel):** Accelerator **GPU T4 x2** (one GPU is used; P100 also works) and **Internet on**
(Kaggle asks for a verified phone number once). Then **Save Version -> Save & Run All (Commit)**: it runs in the
background for up to 12 hours, and this page can be closed.

**What it does:** clones the repository at commit `ef3a158`, installs the pinned libraries (as
`environment-gpu.yml`), rebuilds the DS6 chat subset and checks every data file's SHA-256 against the copy the local
runs use, benchmarks 5 steps, trains (stopping cleanly after `MAX_HOURS`), and evaluates the best adapter on the DS7
test split. T4 and P100 have no native bf16, so the trainer computes in fp16 with loss scaling (recorded in
`meta.json`). Everything worth keeping is written under `/kaggle/working`, the notebook's output.

**Bringing the result home:** download the output (Output tab -> Download), then copy
`runs/qwen3b-explain/` to the project's `runs/qwen3b-explain/` and `reports/tables/assistant_eval_qwen3b.csv` to
`reports/tables/`. The local queue's Qwen steps then finish at once: a run already at its last step trains nothing
and keeps its record.

**If a session ends before training finishes:** add this notebook's previous version output as an input
(Add Input -> Your Work -> this notebook), set `RESUME_FROM` below to its `runs/qwen3b-explain` folder, and run again.
Training continues from the last checkpoint (at most 20 minutes are lost).

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "ef3a158d0cdae3a3c72ba56febc2e0bb98eb2f14"   # the code this run uses; recorded in kaggle_run.json
MAX_HOURS = 11.0       # Kaggle ends a GPU session at 12 h; the trainer saves and stops before that
RESUME_FROM = None     # e.g. "/kaggle/input/<previous-output>/runs/qwen3b-explain" to continue a stopped run
RUN_EVAL = True        # evaluate the best adapter on the DS7 test split once training has finished (about an hour)

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys
from importlib.metadata import version
from pathlib import Path

SRC = Path("/tmp/COS30049A2")             # the code and rebuilt data; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output
RUN = WORK / "runs" / "qwen3b-explain"
PINS = ["transformers==4.52.4", "peft==0.15.2", "bitsandbytes==0.48.1", "accelerate==1.7.0"]  # environment-gpu.yml


def sh(*cmd, log=None, cwd=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)
sh("nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv")

In [ ]:
# DS7 is in the repository; the DS6 chat subset is git-ignored, so it is rebuilt from the same smol-smoltalk file.
# Every file must match the local copy byte for byte, or the two machines would not train on the same data.
sh(sys.executable, "-m", "lm.chat_subset")
EXPECTED = {
    "data/processed/ds7/train.jsonl": "0590c2b4b338c228f6d808c9095eda7fbca5379d995d16e1524c7dc2576c528c",
    "data/processed/ds7/val.jsonl": "7818a89e3ffb8c1698c4342ae43c8182a5ee0174d12adf2b3cde943425b3fdc3",
    "data/processed/ds7/test.jsonl": "c2f1be1d519ad6abe88d051d68e3ecf45369c964578f3eab8c3a723de068f266",
    "data/qwen/chat_train.jsonl": "43a1d891455be4a0b3482999e3301ecc9ede8da5aa0d4dd64c4d8e4045f36361",
    "data/qwen/chat_val.jsonl": "c1710b753740ce469afb1f152b88a1930e52beff23c4d56bd7c1cb4a51e141c1",
    "data/qwen/chat_retention.jsonl": "0f862f4692d1af224c836824a34748176f588dd5aec3eb3c569a6f8bd6794931",
}
for name, digest in EXPECTED.items():
    got = hashlib.sha256((SRC / name).read_bytes()).hexdigest()
    print("same as local" if got == digest else "DIFFERENT", name)
    assert got == digest, f"{name} differs from the local copy; stop and check before training"

# The repository's config, with the run folder moved into the output.
cfg = json.loads((SRC / "configs" / "qwen3b_explain.json").read_text(encoding="utf-8"))
cfg["out_dir"] = str(RUN)
CONFIG = Path("/tmp/qwen3b_kaggle.json")
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
if RESUME_FROM and not (RUN / "state.pt").exists():
    shutil.copytree(RESUME_FROM, RUN, dirs_exist_ok=True)
    print("resuming from", RESUME_FROM)

In [ ]:
# 5 optimizer steps on the longest batches: speed and peak memory, nothing saved (skipped when resuming).
if not (RUN / "state.pt").exists():
    sh(sys.executable, "-m", "lm.qlora", "--config", str(CONFIG), "--bench", log="qwen-bench.log")

In [ ]:
sh(sys.executable, "-m", "lm.qlora", "--config", str(CONFIG), "--max-hours", str(MAX_HOURS), log="qwen-train.log")

In [ ]:
meta = json.loads((RUN / "meta.json").read_text(encoding="utf-8"))
finished = meta["steps_done"] >= meta["max_steps"]
print(f"steps {meta['steps_done']} of {meta['max_steps']}; best validation loss {meta['best_val_loss']:.4f}; "
      f"{meta['tok_per_s']:.0f} tok/s on {meta.get('hardware')}; peak {meta['peak_memory_gb']:.1f} GB")
if not finished:
    print("Not finished: run again with RESUME_FROM set (see the top of the notebook).")
elif RUN_EVAL:
    sh(sys.executable, "-m", "lm.eval_explain", "--model", "qwen3b", "--config", str(CONFIG),
       "--adapter", str(RUN / "adapter_best"), log="eval-qwen.log")
    tables = WORK / "reports" / "tables"
    tables.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / "reports" / "tables" / "assistant_eval_qwen3b.csv", tables)
    shutil.copy(SRC / "reports" / "tables" / "assistant_eval.csv", tables / "assistant_eval_with_qwen3b.csv")

In [ ]:
# What ran, where, with which versions: for the report's reproducibility notes.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
packages = {p: version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")}
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "gpu": gpu, "python": sys.version.split()[0], "packages": packages,
    "finished": finished, "resumed_from": RESUME_FROM,
    "meta": {k: meta.get(k) for k in ("steps_done", "max_steps", "best_val_loss", "tok_per_s", "peak_memory_gb",
                                      "hardware", "fp16_loss_scaling")},
}, indent=2), encoding="utf-8")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")